# Setup

In [ ]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import shutil

# Dusk Crayfish

Loading in sensor data from may fourth 2025 to may 7th 2026 as that is when all data became reliable

In [ ]:
def process_sensor_df(df):
    """Normalize a freshly loaded site frame so it always has a datetime DateTimeUTC."""
    if 'timestamp' in df.columns and 'DateTimeUTC' not in df.columns:
        df = df.rename(columns={'timestamp': 'DateTimeUTC'})
    if 'DateTimeUTC' in df.columns:
        df['DateTimeUTC'] = pd.to_datetime(df['DateTimeUTC'])
    return df


def load_sensor_csv(path):
    """Load a site CSV whether it came out tab or comma separated."""
    df = pd.read_csv(path, sep='\t')
    # the NEW CREEK exports are tabs, but the Raw Creek Data ones are commas. a
    # comma file read as tabs collapses into one fused column whose name still
    # carries the commas, so that is our cue to reread it the right way.
    if df.shape[1] == 1 and ',' in str(df.columns[0]):
        df = pd.read_csv(path, sep=',')
    return process_sensor_df(df)


botanical_garden = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/botanical_garden.csv')
codornices       = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/codornices.csv')
kingman_hall     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/kingman_hall.csv')
north_fork_0     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/north_fork_0.csv')
north_fork_1     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/scnf010.csv')   # weirdly named but this is North Fork 1 (Wickson Footbridge)
oxford           = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/oxford.csv')
south_fork_0     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/south_fork_0.csv')
south_fork_1     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/south_fork_1.csv')
south_fork_2     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/south_fork_2.csv')
south_fork_3     = load_sensor_csv('/content/drive/MyDrive/NEW CREEK/south_fork_3.csv')
university_house = load_sensor_csv('/content/drive/MyDrive/Raw Creek Data/university_house_1778210630544.csv')

# quick peek so we can confirm the columns and format loaded cleanly
print(north_fork_0.head(3))


                                   uuid      site_code         DateTimeUTC  \
0  9f354df0-e9d0-43ed-83a1-29874f270331  North Fork #0 2025-05-31 22:28:00   
1  23be2490-3861-4b93-a41d-ec2d84359906  North Fork #0 2025-05-31 22:38:00   
2  9f7f4b33-5a62-4b5d-84d5-b46fd51cb393  North Fork #0 2025-05-31 22:48:00   

   Meter_Hydros21_Cond  Meter_Hydros21_Depth  Meter_Hydros21_Temp  \
0                559.3                 197.0                16.00   
1                557.3                 196.7                16.00   
2                558.7                 196.3                15.92   

   EnviroDIY_Mayfly_Batt  
0                  4.169  
1                  4.169  
2                  4.169  


# Event windows

One row per event. The event id doubles as the folder name.

In [ ]:
# One row per event. The event id is also the folder name, formatted like the
# old single files (anomaly_DATE_kind_site) where the trailing site marks where
# the anomaly actually happened. Inside each folder we drop one CSV per site
# that has data in that window. We grab every site, no filtering here. Trimming
# down to the validated core (or to 10 nodes later) happens in the tests, not
# in the fixtures, so the folders stay useful when the graph grows.
#
# tz_offset is -8 PST (winter) and -7 PDT (summer). Window times are Pacific
# local; slice_anomaly converts to UTC.

def slice_anomaly(df, start_pt, end_pt, tz_offset):
    """Slice df to a PT window, returned in UTC. tz_offset -8 PST, -7 PDT."""
    start_utc = pd.Timestamp(start_pt) + pd.Timedelta(hours=abs(tz_offset))
    end_utc   = pd.Timestamp(end_pt)   + pd.Timedelta(hours=abs(tz_offset))
    out = df.copy()
    out['DateTimeUTC'] = pd.to_datetime(out['DateTimeUTC'])
    return out[(out['DateTimeUTC'] >= start_utc) & (out['DateTimeUTC'] <= end_utc)].reset_index(drop=True)


# Every site we have, keyed by the name we want the CSV to carry. footbridge is
# the site the raw data loads as north_fork_1 (the scnf010 file), so it gets the
# footbridge.csv name to match what the model calls it. codornices is included
# for completeness since you asked for every site, even though it sits on a
# separate watershed with no graph edges.
all_sites = {
    'footbridge':       north_fork_1,
    'north_fork_0':     north_fork_0,
    'south_fork_0':     south_fork_0,
    'south_fork_1':     south_fork_1,
    'south_fork_2':     south_fork_2,
    'south_fork_3':     south_fork_3,
    'oxford':           oxford,
    'botanical_garden': botanical_garden,
    'kingman_hall':     kingman_hall,
    'university_house': university_house,
    'codornices':       codornices,
}

# id (also the folder name), start_pt, end_pt, tz_offset
events = [
    ('anomaly_2025_05_12_rain_nf1',      '2025-05-10 12:00', '2025-05-14 12:00', -7),
    ('anomaly_2025_06_12_spill_sf',      '2025-06-10 00:00', '2025-06-16 23:59', -7),
    ('anomaly_2025_08_sprinklers_nf0',   '2025-08-01 20:00', '2025-08-03 23:59', -7),
    ('anomaly_2025_09_10_overnight_sf',  '2025-09-08 00:00', '2025-09-12 00:00', -7),
    ('anomaly_2025_11_05_foam_nf1',      '2025-11-03 12:00', '2025-11-07 12:00', -8),
    ('anomaly_2025_11_13_rain_nf1',      '2025-11-11 12:00', '2025-11-15 12:00', -8),
    ('anomaly_2026_01_botanical_actuator', '2026-01-05 00:00', '2026-02-28 23:59', -8),
    ('anomaly_2026_03_20_hydrant_nf0',   '2026-03-20 00:00', '2026-03-21 08:00', -7),
    ('anomaly_2026_04_01_rainfall',      '2026-03-30 00:00', '2026-04-04 00:00', -7),
]

print(f"{len(events)} events, attempting {len(all_sites)} sites each.")


9 events, attempting 11 sites each.


# Weather merge helpers

Fetch Open-Meteo weather and bake it into each fixture, matching production.

In [ ]:
# Pull hourly weather from Open-Meteo archive, same source and columns production
# uses for historical windows. We bake it into each event fixture so the model
# sees the same rain, air temp, and shortwave it was trained on, instead of the
# tests zero-filling those three channels. Rain is an hourly accumulation, so it
# gets split evenly across the sub-hourly creek rows the same way _merge_weather
# does, which keeps windowed rain sums correct. Air temp and shortwave are
# instantaneous, so they copy unchanged across the sub-hourly rows.

_BERKELEY_LAT = 37.873
_BERKELEY_LON = -122.260
_OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
# open-meteo name -> our column name
_OM_VARS = [
    ("temperature_2m",      "air_temp_c"),
    ("precipitation",       "rain_mm"),
    ("shortwave_radiation", "shortwave_radiation"),
]
_ACCUMULATED = {"rain_mm"}  # only rain is an accumulation that needs dividing


def fetch_weather_hourly(start_utc, end_utc):
    """Grab hourly weather for a UTC window. Returns a df indexed by hour, or empty on failure."""
    params = {
        "latitude":   _BERKELEY_LAT,
        "longitude":  _BERKELEY_LON,
        "start_date": pd.Timestamp(start_utc).strftime("%Y-%m-%d"),
        "end_date":   pd.Timestamp(end_utc).strftime("%Y-%m-%d"),
        "hourly":     ",".join(om for om, _ in _OM_VARS),
        "timezone":   "UTC",
    }
    try:
        resp = requests.get(_OPEN_METEO_URL, params=params, timeout=60)
    except requests.exceptions.RequestException as e:
        print(f"    weather fetch failed: {e}")
        return pd.DataFrame()
    if resp.status_code != 200:
        print(f"    weather API returned {resp.status_code}")
        return pd.DataFrame()

    hourly = resp.json().get("hourly", {})
    times = hourly.get("time", [])
    if not times:
        return pd.DataFrame()

    rows = {"datetime": times}
    for om, our in _OM_VARS:
        rows[our] = hourly.get(om, [None] * len(times))
    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).set_index("datetime").sort_index()
    for _, our in _OM_VARS:
        df[our] = pd.to_numeric(df[our], errors="coerce")
    return df


def merge_weather_into_site(site_df, weather_hourly):
    """
    Attach weather to a site slice by flooring each row to the hour and joining.
    site_df is indexed by DateTimeUTC (the sliced fixture). Returns a copy with
    air_temp_c, rain_mm, shortwave_radiation added.
    """
    if weather_hourly.empty:
        for _, our in _OM_VARS:
            site_df[our] = 0.0
        return site_df

    dt = pd.to_datetime(site_df["DateTimeUTC"], utc=True)
    rows_per_hour = 4  # creek data is 15-min, so 4 rows per hour

    wh = weather_hourly.copy()
    for our in _OM_VARS:
        col = our[1]
        if col in _ACCUMULATED and col in wh.columns:
            wh[col] = wh[col] / rows_per_hour  # split hourly rain across the 4 sub-hourly rows

    hour_key = dt.dt.floor("h")
    merged = site_df.copy()
    for _, our in _OM_VARS:
        if our in wh.columns:
            merged[our] = hour_key.map(wh[our]).fillna(0.0).values
        else:
            merged[our] = 0.0
    return merged


print("Weather helpers ready.")


Weather helpers ready.


# Build event folders

One folder per event, one CSV per site that has data in the window.

In [ ]:
# Each event becomes a folder named after the event. Inside, one CSV per site
# that has rows in the window, now with weather columns merged in so the fixtures
# carry the same rain, air temp, and shortwave the model trained on. Sites with
# no data in the window are reported and skipped.
output_root = '/content/event_data'
os.makedirs(output_root, exist_ok=True)

keep_cols = ['DateTimeUTC', 'Meter_Hydros21_Cond', 'Meter_Hydros21_Depth', 'Meter_Hydros21_Temp']

manifest = []

for event_id, start_pt, end_pt, tz_offset in events:
    event_dir = os.path.join(output_root, event_id)
    os.makedirs(event_dir, exist_ok=True)
    print(f"\n=== {event_id} ===")

    # One weather pull per event window, shared across all its sites. Widen the
    # fetch by a day on each side so hour-flooring near the edges still lands.
    start_utc = pd.Timestamp(start_pt) + pd.Timedelta(hours=abs(tz_offset)) - pd.Timedelta(days=1)
    end_utc   = pd.Timestamp(end_pt)   + pd.Timedelta(hours=abs(tz_offset)) + pd.Timedelta(days=1)
    weather_hourly = fetch_weather_hourly(start_utc, end_utc)
    if weather_hourly.empty:
        print("  (weather unavailable for this window, sites will get zero weather)")

    for site_name, site_df in all_sites.items():
        sliced = slice_anomaly(site_df, start_pt, end_pt, tz_offset)
        present = [c for c in keep_cols if c in sliced.columns]
        sliced = sliced[present]
        rows = len(sliced)

        if rows == 0:
            print(f"  {site_name:17s}     0 rows  (no data this window, skipped)")
            manifest.append({'event': event_id, 'site': site_name, 'rows': 0, 'written': False})
            continue

        sliced = merge_weather_into_site(sliced, weather_hourly)

        fpath = os.path.join(event_dir, f"{site_name}.csv")
        sliced.to_csv(fpath, index=False)
        print(f"  {site_name:17s} {rows:>5} rows  ->  {event_id}/{site_name}.csv")
        manifest.append({'event': event_id, 'site': site_name, 'rows': rows, 'written': True})

print(f"\nDone. Folders written under {output_root}")



=== anomaly_2025_05_12_rain_nf1 ===
  footbridge          205 rows  ->  anomaly_2025_05_12_rain_nf1/footbridge.csv
  north_fork_0        521 rows  ->  anomaly_2025_05_12_rain_nf1/north_fork_0.csv
  south_fork_0          0 rows  (no data this window, skipped)
  south_fork_1          0 rows  (no data this window, skipped)
  south_fork_2          0 rows  (no data this window, skipped)
  south_fork_3          0 rows  (no data this window, skipped)
  oxford                0 rows  (no data this window, skipped)
  botanical_garden      0 rows  (no data this window, skipped)
  kingman_hall          0 rows  (no data this window, skipped)
  university_house      0 rows  (no data this window, skipped)
  codornices            0 rows  (no data this window, skipped)

=== anomaly_2025_06_12_spill_sf ===
  footbridge           28 rows  ->  anomaly_2025_06_12_spill_sf/footbridge.csv
  north_fork_0        558 rows  ->  anomaly_2025_06_12_spill_sf/north_fork_0.csv
  south_fork_0        559 rows  ->  ano

# Coverage check

Which sites landed in each event folder.

In [ ]:
# Which sites landed in each event folder. A blank cell means that site had no
# data in the window and got no file. This is the honest coverage picture you
# carry into the tests.
man = pd.DataFrame(manifest)
pivot = man.pivot_table(index='event', columns='site', values='rows', fill_value=0)
print("Rows per site per event (0 = no file written):\n")
print(pivot.to_string())


Rows per site per event (0 = no file written):

site                                botanical_garden  codornices  footbridge  kingman_hall  north_fork_0  oxford  south_fork_0  south_fork_1  south_fork_2  south_fork_3  university_house
event                                                                                                                                                                                     
anomaly_2025_05_12_rain_nf1                      0.0         0.0       205.0           0.0         521.0     0.0           0.0           0.0           0.0           0.0               0.0
anomaly_2025_06_12_spill_sf                      0.0       672.0        28.0           0.0         558.0   672.0         559.0         559.0         558.0         553.0             470.0
anomaly_2025_08_sprinklers_nf0                   0.0       208.0         7.0           0.0         206.0   208.0           2.0         208.0         207.0           0.0             208.0
anomaly_2025_09_1

# Download

Zip the event folders and pull them into the repo.

In [ ]:
# Zip the whole event_data tree and download. Unzip into the repo so each event
# folder lands wherever the rewritten tests look for it.
shutil.make_archive('/content/event_data', 'zip', '/content/event_data')
files.download('/content/event_data.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sf = pd.read_csv('event_data/anomaly_2025_09_10_overnight_sf/south_fork_1.csv')
print(sf['rain_mm'].describe())
print(sf['rain_mm'].sum())

count    385.000000
mean       0.008571
std        0.021300
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.125000
Name: rain_mm, dtype: float64
3.3


In [ ]:
ap = pd.read_csv('event_data/anomaly_2026_04_01_rainfall/north_fork_0.csv')
print(ap['rain_mm'].describe())
print(ap['rain_mm'].sum())

count    481.000000
mean       0.049688
std        0.122644
min        0.000000
25%        0.000000
50%        0.000000
75%        0.025000
max        0.675000
Name: rain_mm, dtype: float64
23.9
